# Mosquito Flight Trajectory Visualization

이 노트북은 모기의 3차원 비행 궤적을 시각화하기 위해 작성되었습니다.
`data/train` 폴더 내의 특정 CSV 파일을 지정하여 모기의 위치(x, y, z)를 시간(timestep_ms)에 따라 점과 선으로 연결하여 보여줍니다.
또한 `train_labels.csv`가 존재할 경우, 예측해야 할 최종 타겟 포인트를 함께 표시합니다.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import os

## 1. 시각화할 파일 지정

아래 `FILE_NAME` 변수에 시각화하고 싶은 파일명을 입력하세요.

In [ ]:
# 시각화하고 싶은 파일 이름을 입력하세요 (예: 'TRAIN_00001.csv')
FILE_NAME = 'TRAIN_00001.csv'
DATA_DIR = 'data/train'
LABELS_PATH = 'data/train_labels.csv'

file_path = os.path.join(DATA_DIR, FILE_NAME)
sample_id = FILE_NAME.split('.')[0]

if os.path.exists(file_path):
    print(f"파일을 찾았습니다: {file_path}")
    df = pd.read_csv(file_path)
    
    # 라벨 정보가 있는지 확인
    target_point = None
    if os.path.exists(LABELS_PATH):
        labels_df = pd.read_csv(LABELS_PATH)
        label_row = labels_df[labels_df['id'] == sample_id]
        if not label_row.empty:
            target_point = label_row.iloc[0]
            print(f"타겟 포인트를 찾았습니다: x={target_point['x']:.4f}, y={target_point['y']:.4f}, z={target_point['z']:.4f}")
    
    display(df.head())
else:
    print(f"에러: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {file_path}")

## 2. 3차원 궤적 시각화

Plotly를 사용하여 인터랙티브한 3D 그래프를 생성합니다. 
- **파란색 선/점**: 모기의 비행 궤적 (Viridis 색상 축은 시간을 나타냄)
- **빨간색 X**: 예측해야 할 타겟 포인트 (정답)

In [ ]:
def plot_mosquito_trajectory(df, target_point=None, title=""):
    # 1. 궤적 선과 점 생성
    trace_trajectory = go.Scatter3d(
        x=df['x'],
        y=df['y'],
        z=df['z'],
        mode='lines+markers',
        name='Trajectory',
        marker=dict(
            size=4,
            color=df['timestep_ms'],
            colorscale='Viridis',
            opacity=0.8,
            colorbar=dict(title="ms", thickness=20, x=1.1)
        ),
        line=dict(color='darkblue', width=2),
        text=df['timestep_ms'],
        hoverinfo='text+x+y+z'
    )

    data = [trace_trajectory]

    # 2. 타겟 포인트 추가 (있는 경우)
    if target_point is not None:
        trace_target = go.Scatter3d(
            x=[target_point['x']],
            y=[target_point['y']],
            z=[target_point['z']],
            mode='markers',
            name='Target (Label)',
            marker=dict(size=8, color='red', symbol='cross', opacity=1.0),
            hoverinfo='name+x+y+z'
        )
        data.append(trace_target)

    # 레이아웃 설정
    fig = go.Figure(data=data)
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data' 
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    fig.show()

if 'df' in locals():
    target = target_point if 'target_point' in locals() else None
    plot_mosquito_trajectory(df, target, f"Mosquito Flight Trajectory: {FILE_NAME}")